# Kyoto Tourism Redistribution — Data Ingestion Pipeline
**Crux AI | Japan Tourism ML Research**

This notebook collects and structures live data from five sources:
1. **Geoapify Places API** — Kyoto location discovery (names, coordinates, categories, opening hours)
2. **Overpass API (OpenStreetMap)** — Wikidata IDs, OSM tags, and richer metadata
3. **Wikipedia Pageviews API** — Article pageview counts as a popularity proxy
4. **Japan Meteorological Agency (JMA)** — Kyoto weather forecasts
5. **JNTO Public Statistics** — Monthly visitor volume by nationality

All data is saved as structured CSVs ready for LightGBM training.


In [1]:
# DEPENDENCIES
import requests
import pandas as pd
import numpy as np
import json
import time
import os
from datetime import datetime

print('All imports loaded.')

All imports loaded.


In [2]:
# CONFIG
from dotenv import load_dotenv
load_dotenv()
GEOAPIFY_API_KEY = os.getenv("GEOAPIFY_API_KEY")

OUTPUT_DIR    = "./kyoto_data"
JMA_AREA_CODE = "260000"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Kyoto bounding box — Geoapify rect format: lon1,lat1,lon2,lat2
KYOTO_BBOX = "135.6800,34.9200,135.8200,35.1000"

# Overpass uses (south, west, north, east)
KYOTO_BBOX_OVERPASS = (34.9200, 135.6800, 35.1000, 135.8200)

# Geoapify category codes
# Full list: https://apidocs.geoapify.com/docs/places/#categories
GEOAPIFY_CATEGORIES = [
    "tourism.sights",
    "tourism.attraction",
    "entertainment.culture",
    "entertainment.museum",
    "religion",
    "natural"
]

print(f"Config loaded. Output: {OUTPUT_DIR}")


Config loaded. Output: ./kyoto_data


---
## Module 1A: Geoapify - Location Discovery

Fetches all tourist-relevant named locations within Kyoto's bounding box.
Geoapify is backed by OpenStreetMap and returns structured opening hours.

**Output:** `geoapify_locations.csv`
```
place_id | name | lat | lng | category | subcategory |
address | opening_hours | website | fetched_at
```

In [3]:
def fetch_geoapify_places(api_key, categories, bbox, limit=500):
    url = "https://api.geoapify.com/v2/places"
    all_places = []
    seen_ids = set()

    for category in categories:
        print(f"  Fetching: {category}")
        offset = 0
        page_limit = 100

        while True:
            params = {
                "categories": category,
                "filter": f"rect:{bbox}",
                "limit": page_limit,
                "offset": offset,
                "apiKey": api_key
            }
            try:
                r = requests.get(url, params=params, timeout=15)
                r.raise_for_status()
                data = r.json()
            except Exception as e:
                print(f"    Error on {category} offset {offset}: {e}")
                break

            features = data.get("features", [])
            if not features:
                break

            for feat in features:
                props = feat.get("properties", {})
                place_id = props.get("place_id")
                if not place_id or place_id in seen_ids:
                    continue
                seen_ids.add(place_id)

                coords = feat.get("geometry", {}).get("coordinates", [None, None])
                opening_hours = props.get("opening_hours", None)

                all_places.append({
                    "place_id":      place_id,
                    "name":          props.get("name"),
                    "lat":           coords[1],
                    "lng":           coords[0],
                    "category":      category,
                    "subcategory":   props.get("categories", [None])[-1] if props.get("categories") else None,
                    "address":       props.get("formatted"),
                    "city":          props.get("city"),
                    "district":      props.get("district"),
                    "opening_hours": json.dumps(opening_hours) if opening_hours else None,
                    "website":       props.get("website"),
                    "fetched_at":    datetime.utcnow().isoformat()
                })

            offset += page_limit
            if offset >= limit or len(features) < page_limit:
                break
            time.sleep(0.3)

        print(f"    Unique locations so far: {len(all_places)}")
        time.sleep(0.5)

    return pd.DataFrame(all_places)


print("Fetching Kyoto locations from Geoapify...")
df_locations = fetch_geoapify_places(GEOAPIFY_API_KEY, GEOAPIFY_CATEGORIES, KYOTO_BBOX)
if df_locations.empty:
    raise SystemExit("⚠ Geoapify returned 0 locations. Check your API key in .env")
df_locations = df_locations.dropna(subset=["name"]).reset_index(drop=True)

geo_path = f"{OUTPUT_DIR}/geoapify_locations.csv"
df_locations.to_csv(geo_path, index=False)

print(f"\nSaved {len(df_locations)} named locations to {geo_path}")
df_locations[["name", "lat", "lng", "category", "opening_hours"]].head(10)

Fetching Kyoto locations from Geoapify...
  Fetching: tourism.sights


C:\Users\rigve\AppData\Local\Temp\ipykernel_5160\3812054265.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at":    datetime.utcnow().isoformat()


    Unique locations so far: 500
  Fetching: tourism.attraction
    Unique locations so far: 988
  Fetching: entertainment.culture
    Unique locations so far: 1047
  Fetching: entertainment.museum
    Unique locations so far: 1160
  Fetching: religion
    Unique locations so far: 1642
  Fetching: natural
    Unique locations so far: 2141

Saved 1720 named locations to ./kyoto_data/geoapify_locations.csv


,name,lat,lng,category,opening_hours
0,二条城,35.014008,135.748537,tourism.sights,"""Mo-Su 08:45-17:00"""
1,仁和寺,35.029812,135.714150,tourism.sights,"""Mar-Nov Mo-Su 09:00-17:00; Dec-Feb Mo-Su 09:0..."
2,龍安寺,35.033463,135.718088,tourism.sights,"""Mo-Su 08:00-17:00"""
3,上賀茂神社,35.060468,135.752259,tourism.sights,"""Mo-Su 08:30-17:00"""
4,下鴨神社,35.037010,135.772515,tourism.sights,"""Mo-Su 05:30-18:00"""
5,金閣寺,35.039529,135.729537,tourism.sights,"""Mo-Su 09:00-17:00"""
6,金堂,34.980356,135.747702,tourism.sights,"""Mo-Su 08:00-17:00"""
7,清水寺,34.994303,135.784439,tourism.sights,"""Mo-Su 06:00-18:00"""
8,東寺,34.980631,135.747719,tourism.sights,"""Mo-Su 05:00-17:00"""
9,西本願寺,34.991832,135.751046,tourism.sights,"""Mo-Su 05:30-17:00"""


---
## Module 1B: Overpass API — OSM Enrichment & Wikidata IDs

Queries OpenStreetMap via the Overpass API for all tourism/historic/religious/natural
features in Kyoto. Extracts Wikidata IDs and richer tags, then matches to Geoapify
locations by proximity (< 50m haversine).

**No API key required.** Free public endpoint.

**Output:** `overpass_enrichment.csv`
```
place_id | wikidata_id | wikipedia_tag | osm_tourism |
osm_historic | name_en | heritage
```


In [4]:
import math

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def haversine_m(lat1, lon1, lat2, lon2):
    """Haversine distance in meters between two points."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


def fetch_overpass_kyoto(bbox):
    """
    Fetches all tourism/historic/religious/natural features with
    wikidata tags from Overpass API.
    """
    south, west, north, east = bbox
    query = f"""
    [out:json][timeout:120];
    (
      nwr["tourism"]["wikidata"]({south},{west},{north},{east});
      nwr["historic"]["wikidata"]({south},{west},{north},{east});
      nwr["amenity"="place_of_worship"]["wikidata"]({south},{west},{north},{east});
      nwr["leisure"="park"]["wikidata"]({south},{west},{north},{east});
      nwr["building"]["wikidata"]["tourism"]({south},{west},{north},{east});
    );
    out center tags;
    """
    print("Querying Overpass API...")
    r = requests.post(OVERPASS_URL, data={"data": query}, timeout=120)
    r.raise_for_status()
    elements = r.json().get("elements", [])
    print(f"  Received {len(elements)} OSM elements")

    rows = []
    seen_wd = set()
    for el in elements:
        tags = el.get("tags", {})
        wd = tags.get("wikidata")
        if not wd or wd in seen_wd:
            continue
        seen_wd.add(wd)

        # Get coordinates (center for ways/relations)
        if el["type"] == "node":
            lat, lon = el.get("lat"), el.get("lon")
        else:
            c = el.get("center", {})
            lat, lon = c.get("lat"), c.get("lon")
        if lat is None or lon is None:
            continue

        rows.append({
            "osm_id":        el.get("id"),
            "osm_type":      el.get("type"),
            "osm_lat":       lat,
            "osm_lon":       lon,
            "wikidata_id":   wd,
            "wikipedia_tag": tags.get("wikipedia"),
            "name_osm":      tags.get("name"),
            "name_en":       tags.get("name:en"),
            "osm_tourism":   tags.get("tourism"),
            "osm_historic":  tags.get("historic"),
            "heritage":      tags.get("heritage"),
        })

    return pd.DataFrame(rows)


def match_overpass_to_geoapify(df_geo, df_osm, max_dist_m=50):
    """
    For each Geoapify location, find the closest OSM feature within max_dist_m.
    Returns df_geo with Overpass columns joined.
    """
    matches = []
    osm_lats = df_osm["osm_lat"].values
    osm_lons = df_osm["osm_lon"].values

    for _, row in df_geo.iterrows():
        best_idx, best_dist = None, max_dist_m
        for j in range(len(df_osm)):
            d = haversine_m(row["lat"], row["lng"], osm_lats[j], osm_lons[j])
            if d < best_dist:
                best_dist = d
                best_idx = j
        matches.append(best_idx)

    # Build enrichment columns
    enrich_cols = ["wikidata_id", "wikipedia_tag", "name_en",
                   "osm_tourism", "osm_historic", "heritage"]
    for col in enrich_cols:
        df_geo[col] = [
            df_osm.iloc[idx][col] if idx is not None else None
            for idx in matches
        ]
    df_geo["osm_match_dist_m"] = [
        round(haversine_m(df_geo.iloc[i]["lat"], df_geo.iloc[i]["lng"],
                          osm_lats[idx], osm_lons[idx]), 1)
        if idx is not None else None
        for i, idx in enumerate(matches)
    ]
    return df_geo


# Fetch Overpass data
df_osm = fetch_overpass_kyoto(KYOTO_BBOX_OVERPASS)
print(f"Unique Wikidata entries: {len(df_osm)}")

# Load Geoapify locations and match
df_geo = pd.read_csv(f"{OUTPUT_DIR}/geoapify_locations.csv")
print(f"\nMatching {len(df_geo)} Geoapify locations to OSM (< 50m)...")
df_enriched = match_overpass_to_geoapify(df_geo.copy(), df_osm)

matched = df_enriched["wikidata_id"].notna().sum()
print(f"Matched: {matched} / {len(df_enriched)} locations have Wikidata IDs")

enrich_path = f"{OUTPUT_DIR}/overpass_enrichment.csv"
df_enriched.to_csv(enrich_path, index=False)
print(f"Saved to {enrich_path}")

df_enriched[df_enriched["wikidata_id"].notna()][
    ["name", "wikidata_id", "name_en", "osm_tourism", "osm_historic"]
].head(15)


Querying Overpass API...
  Received 549 OSM elements
Unique Wikidata entries: 540

Matching 1720 Geoapify locations to OSM (< 50m)...
Matched: 711 / 1720 locations have Wikidata IDs
Saved to ./kyoto_data/overpass_enrichment.csv


,name,wikidata_id,name_en,osm_tourism,osm_historic
0,二条城,Q1013399,Nijō Castle,attraction,castle
1,仁和寺,Q1202871,Ninna temple,attraction,heritage
2,龍安寺,Q587371,Ryōan-ji,attraction,heritage
3,上賀茂神社,Q700448,Kamigamo Shrine,NaN,heritage
4,下鴨神社,Q701620,Shimogamo Shrine,NaN,monastery
5,金閣寺,Q270983,Kinkaku-ji,NaN,heritage
6,金堂,Q107020573,Golden Hall,NaN,yes
7,清水寺,Q221716,Kiyomizu-dera,attraction,yes
8,東寺,Q107020573,Golden Hall,NaN,yes
9,西本願寺,Q1146038,Nishi-Hongan-ji,NaN,heritage


---
## Module 1C: Wikipedia Pageviews — Popularity Signal

Uses Wikidata IDs from Overpass to resolve English + Japanese Wikipedia article titles,
then fetches 12-month pageview totals. Combined pageviews are min-max scaled to a
0–1 `popularity` score.

**No API key required.** Free Wikimedia REST API.

**Output:** `wikipedia_popularity.csv`
```
place_id | wikidata_id | wiki_en | wiki_ja |
pageviews_en_12m | pageviews_ja_12m | wiki_pageviews_12m | popularity
```


In [5]:
from datetime import datetime, timedelta
from urllib.parse import quote

WIKIDATA_API  = "https://www.wikidata.org/w/api.php"
WIKI_PV_URL   = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
WIKI_HEADERS  = {"User-Agent": "KyotoDSS/1.0 (tourism-research; python-requests)"}

# Date range: last 12 full months
end_date   = datetime.now().replace(day=1) - timedelta(days=1)  # last day of prev month
start_date = end_date.replace(year=end_date.year - 1, day=1)    # 12 months back
PV_START = start_date.strftime("%Y%m%d")
PV_END   = end_date.strftime("%Y%m%d")
print(f"Pageview window: {PV_START} → {PV_END}")


def resolve_wiki_titles(wikidata_ids, batch_size=50):
    """
    Resolves Wikidata IDs to English and Japanese Wikipedia article titles
    using the Wikidata API in batches.
    """
    result = {}  # wd_id → {"en": title, "ja": title}
    ids = list(wikidata_ids)

    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "sitelinks",
            "sitefilter": "enwiki|jawiki",
            "format": "json"
        }
        try:
            r = requests.get(WIKIDATA_API, params=params,
                             headers=WIKI_HEADERS, timeout=15)
            r.raise_for_status()
            entities = r.json().get("entities", {})
            for wd_id, ent in entities.items():
                sitelinks = ent.get("sitelinks", {})
                result[wd_id] = {
                    "en": sitelinks.get("enwiki", {}).get("title"),
                    "ja": sitelinks.get("jawiki", {}).get("title"),
                }
        except Exception as e:
            print(f"  Wikidata batch error: {e}")
        time.sleep(0.2)

    return result


def fetch_pageviews(project, title, start, end):
    """
    Fetches monthly pageviews for a single Wikipedia article.
    Returns total views or 0 on error.
    """
    encoded_title = quote(title.replace(" ", "_"), safe="")
    url = f"{WIKI_PV_URL}/{project}/all-access/user/{encoded_title}/monthly/{start}/{end}"
    try:
        r = requests.get(url, headers=WIKI_HEADERS, timeout=10)
        if r.status_code == 200:
            items = r.json().get("items", [])
            return sum(item.get("views", 0) for item in items)
    except Exception:
        pass
    return 0


# Load enriched data
df_enriched = pd.read_csv(f"{OUTPUT_DIR}/overpass_enrichment.csv")
df_with_wd = df_enriched[df_enriched["wikidata_id"].notna()].copy()
unique_wds = df_with_wd["wikidata_id"].unique()
print(f"Resolving {len(unique_wds)} Wikidata IDs to Wikipedia titles...")

# Step 1: Resolve Wikidata → Wikipedia titles
wd_titles = resolve_wiki_titles(unique_wds)
en_count = sum(1 for v in wd_titles.values() if v.get("en"))
ja_count = sum(1 for v in wd_titles.values() if v.get("ja"))
print(f"  Resolved: {en_count} English, {ja_count} Japanese Wikipedia articles")

# Step 2: Fetch pageviews
print(f"\nFetching pageviews for {len(wd_titles)} entities...")
pv_records = []
for idx, wd_id in enumerate(wd_titles):
    titles = wd_titles[wd_id]
    en_title = titles.get("en")
    ja_title = titles.get("ja")

    pv_en = fetch_pageviews("en.wikipedia", en_title, PV_START, PV_END) if en_title else 0
    pv_ja = fetch_pageviews("ja.wikipedia", ja_title, PV_START, PV_END) if ja_title else 0

    pv_records.append({
        "wikidata_id":       wd_id,
        "wiki_en":           en_title,
        "wiki_ja":           ja_title,
        "pageviews_en_12m":  pv_en,
        "pageviews_ja_12m":  pv_ja,
        "wiki_pageviews_12m": pv_en + pv_ja,
    })

    if (idx + 1) % 50 == 0:
        print(f"  [{idx+1}/{len(wd_titles)}] processed")
    time.sleep(0.1)  # polite rate limit

df_pv = pd.DataFrame(pv_records)

# Min-max scale to popularity 0–1
max_pv = df_pv["wiki_pageviews_12m"].max()
df_pv["popularity"] = (df_pv["wiki_pageviews_12m"] / max_pv).round(4) if max_pv > 0 else 0

pv_path = f"{OUTPUT_DIR}/wikipedia_popularity.csv"
df_pv.to_csv(pv_path, index=False)

print(f"\nSaved {len(df_pv)} records to {pv_path}")
print(f"Top pageview locations:")
df_pv.sort_values("wiki_pageviews_12m", ascending=False)[
    ["wiki_en", "wiki_ja", "wiki_pageviews_12m", "popularity"]
].head(15)


Pageview window: 20250201 → 20260228
Resolving 439 Wikidata IDs to Wikipedia titles...
  Resolved: 143 English, 377 Japanese Wikipedia articles

Fetching pageviews for 439 entities...
  [50/439] processed
  [100/439] processed
  [150/439] processed
  [200/439] processed
  [250/439] processed
  [300/439] processed
  [350/439] processed
  [400/439] processed

Saved 439 records to ./kyoto_data/wikipedia_popularity.csv
Top pageview locations:


,wiki_en,wiki_ja,wiki_pageviews_12m,popularity
37,Emperor Tenji,天智天皇,362991,1.0000
7,Kiyomizu-dera,清水寺,333859,0.9197
5,Kinkaku-ji,鹿苑寺,330539,0.9106
76,Fushimi Inari-taisha,伏見稲荷大社,329544,0.9079
349,Emperor Sutoku,崇徳天皇,324116,0.8929
0,Nijō Castle,二条城,241131,0.6643
51,Empress Shōken,昭憲皇太后,202745,0.5585
9,Kyoto Imperial Palace,京都御所,174125,0.4797
55,Fushimi Castle,伏見城,154403,0.4254
207,Sanjūsangen-dō,三十三間堂,133908,0.3689


---
## Module 1D: Merge Geoapify + Overpass + Wikipedia

Joins location metadata with OSM enrichment and Wikipedia popularity into one master table.


In [6]:
df_geo = pd.read_csv(f"{OUTPUT_DIR}/overpass_enrichment.csv")
df_pv  = pd.read_csv(f"{OUTPUT_DIR}/wikipedia_popularity.csv")

# Merge popularity by wikidata_id
pv_cols = ["wikidata_id", "wiki_en", "wiki_ja",
           "pageviews_en_12m", "pageviews_ja_12m",
           "wiki_pageviews_12m", "popularity"]
pv_cols = [c for c in pv_cols if c in df_pv.columns]

df_master = df_geo.merge(df_pv[pv_cols], on="wikidata_id", how="left",
                         suffixes=("", "_wiki"))

# Use wiki popularity, fill 0 for unmatched
if "popularity_wiki" in df_master.columns:
    df_master["popularity"] = df_master["popularity_wiki"].fillna(0)
    df_master.drop(columns=["popularity_wiki"], inplace=True)
elif "popularity" not in df_master.columns:
    df_master["popularity"] = 0
else:
    df_master["popularity"] = df_master["popularity"].fillna(0)

df_master["wiki_pageviews_12m"] = df_master["wiki_pageviews_12m"].fillna(0).astype(int)
df_master["has_wiki_data"]      = df_master["wikidata_id"].notna().astype(int)

master_path = f"{OUTPUT_DIR}/locations_master.csv"
df_master.to_csv(master_path, index=False)

print(f"Master table: {len(df_master)} locations")
print(f"With Wikipedia data: {df_master['has_wiki_data'].sum()}")
print(f"Saved to {master_path}")
print("\nPopularity distribution (matched only):")
print(df_master[df_master["has_wiki_data"]==1]["popularity"].describe().round(3))
df_master[["name", "wikidata_id", "name_en", "popularity",
           "wiki_pageviews_12m", "opening_hours"]].head(10)


Master table: 1720 locations
With Wikipedia data: 711
Saved to ./kyoto_data/locations_master.csv

Popularity distribution (matched only):
count    711.000
mean       0.050
std        0.136
min        0.000
25%        0.003
50%        0.010
75%        0.030
max        1.000
Name: popularity, dtype: float64


,name,wikidata_id,name_en,popularity,wiki_pageviews_12m,opening_hours
0,二条城,Q1013399,Nijō Castle,0.6643,241131,"""Mo-Su 08:45-17:00"""
1,仁和寺,Q1202871,Ninna temple,0.2046,74274,"""Mar-Nov Mo-Su 09:00-17:00; Dec-Feb Mo-Su 09:0..."
2,龍安寺,Q587371,Ryōan-ji,0.2783,101008,"""Mo-Su 08:00-17:00"""
3,上賀茂神社,Q700448,Kamigamo Shrine,0.1742,63232,"""Mo-Su 08:30-17:00"""
4,下鴨神社,Q701620,Shimogamo Shrine,0.1984,72028,"""Mo-Su 05:30-18:00"""
5,金閣寺,Q270983,Kinkaku-ji,0.9106,330539,"""Mo-Su 09:00-17:00"""
6,金堂,Q107020573,Golden Hall,0.0000,0,"""Mo-Su 08:00-17:00"""
7,清水寺,Q221716,Kiyomizu-dera,0.9197,333859,"""Mo-Su 06:00-18:00"""
8,東寺,Q107020573,Golden Hall,0.0000,0,"""Mo-Su 05:00-17:00"""
9,西本願寺,Q1146038,Nishi-Hongan-ji,0.2767,100439,"""Mo-Su 05:30-17:00"""


---
## Module 2: JMA Weather Forecast

Free public API, no key required.

**Output:** `weather_forecast.csv`

In [7]:
def fetch_jma_forecast(area_code):
    url = f"https://www.jma.go.jp/bosai/forecast/data/forecast/{area_code}.json"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    records = []
    fetch_time = datetime.utcnow().isoformat()

    for block in data:
        office = block.get("publishingOffice", "")
        for ts in block.get("timeSeries", []):
            time_defines = ts.get("timeDefines", [])
            for area in ts.get("areas", []):
                area_name  = area.get("area", {}).get("name", "")
                codes      = area.get("weatherCodes", [])
                weathers   = area.get("weathers", [])
                pops       = area.get("pops", [])
                temps_min  = area.get("tempsMin", [])
                temps_max  = area.get("tempsMax", [])

                for i, td in enumerate(time_defines):
                    records.append({
                        "date":              td[:10],
                        "datetime":          td,
                        "area":              area_name,
                        "publishing_office": office,
                        "weather_code":      codes[i] if i < len(codes) else None,
                        "weather_desc":      weathers[i].replace("\u3000", " ").strip() if i < len(weathers) else None,
                        "pop":               pops[i] if i < len(pops) else None,
                        "temp_min":          temps_min[i] if i < len(temps_min) else None,
                        "temp_max":          temps_max[i] if i < len(temps_max) else None,
                        "fetched_at":        fetch_time
                    })

    return pd.DataFrame(records)


print("Fetching JMA Kyoto forecast...")
df_weather = fetch_jma_forecast(JMA_AREA_CODE)
df_weather = df_weather.dropna(subset=["weather_code"]).drop_duplicates(subset=["date", "area", "weather_code"])

weather_path = f"{OUTPUT_DIR}/weather_forecast.csv"
df_weather.to_csv(weather_path, index=False)

print(f"Saved {len(df_weather)} records to {weather_path}")
df_weather[["date", "area", "weather_desc", "pop", "temp_min", "temp_max"]].head(10)

Fetching JMA Kyoto forecast...
Saved 16 records to ./kyoto_data/weather_forecast.csv


C:\Users\rigve\AppData\Local\Temp\ipykernel_5160\3007798907.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  fetch_time = datetime.utcnow().isoformat()


,date,area,weather_desc,pop,temp_min,temp_max
0,2026-03-08,南部,くもり 夕方 から 晴れ,NaN,NaN,NaN
1,2026-03-09,南部,晴れ 夕方 から くもり 所により 夜遅く 雪か雨,NaN,NaN,NaN
2,2026-03-10,南部,くもり 時々 晴れ,NaN,NaN,NaN
3,2026-03-08,北部,くもり,NaN,NaN,NaN
4,2026-03-09,北部,くもり 時々 晴れ 夜遅く 雨か雪 所により 夜 雷 を伴う,NaN,NaN,NaN
5,2026-03-10,北部,くもり 一時 雪か雨,NaN,NaN,NaN
28,2026-03-11,南部,NaN,20,NaN,NaN
29,2026-03-12,南部,NaN,30,NaN,NaN
30,2026-03-13,南部,NaN,30,NaN,NaN
31,2026-03-14,南部,NaN,20,NaN,NaN


---
## Module 3: JNTO Visitor Statistics

Historical monthly visitor counts by nationality (2003-present).
Used to build seasonal index as a LightGBM feature.

**Output:** `jnto_visitor_stats.csv`, `seasonal_index.csv`

In [10]:
from io import BytesIO

def fetch_jnto_statistics():
    """
    Fetches JNTO visitor statistics from official Excel file.
    The file has separate sheets per year (2003-2026), each with
    monthly visitor counts by nationality.
    """
    # URL from https://www.jnto.go.jp/statistics/data/visitors-statistics/
    url = "https://www.jnto.go.jp/statistics/data/_files/20260218_1615-5.xlsx"
    print("Downloading JNTO Excel...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    print(f"Download OK ({len(r.content):,} bytes)")

    xls = pd.ExcelFile(BytesIO(r.content))
    print(f"Sheets found: {xls.sheet_names}")

    all_records = []
    month_map = {
        '1月': 1, '2月': 2, '3月': 3, '4月': 4, '5月': 5, '6月': 6,
        '7月': 7, '8月': 8, '9月': 9, '10月': 10, '11月': 11, '12月': 12
    }

    for sheet_name in xls.sheet_names:
        try:
            year = int(sheet_name)
        except ValueError:
            continue

        # Read with header=3 (row 3 has month names: 1月, 伸率, 2月, ...)
        df = pd.read_excel(xls, sheet_name=sheet_name, header=3)
        # First column is nationality/region name
        nat_col = df.columns[0]
        df = df.rename(columns={nat_col: 'nationality'})
        df = df.dropna(subset=['nationality'])

        # Extract only month columns (skip growth-rate '伸率' columns)
        for col in df.columns:
            col_str = str(col).strip()
            if col_str in month_map:
                month = month_map[col_str]
                for _, row in df.iterrows():
                    val = pd.to_numeric(row[col], errors='coerce')
                    if pd.notna(val) and val > 0:
                        all_records.append({
                            'year': year,
                            'month': month,
                            'nationality': str(row['nationality']).strip(),
                            'visitor_count': val
                        })

    df_out = pd.DataFrame(all_records)
    df_out['fetched_at'] = datetime.now().isoformat()
    return df_out


df_jnto = fetch_jnto_statistics()
jnto_path = f"{OUTPUT_DIR}/jnto_visitor_stats.csv"
df_jnto.to_csv(jnto_path, index=False)

print(f"Saved {len(df_jnto)} records | Year range: {df_jnto['year'].min()} - {df_jnto['year'].max()}")

# Seasonal index (exclude COVID years)
df_recent = df_jnto[(df_jnto["year"] >= 2015) & (~df_jnto["year"].isin([2020, 2021]))]
df_monthly = df_recent.groupby(["year", "month"])["visitor_count"].sum().reset_index()
monthly_avg = df_monthly.groupby("month")["visitor_count"].mean()
seasonal_index = (monthly_avg / monthly_avg.mean()).reset_index()
seasonal_index.columns = ["month", "seasonal_index"]

seasonal_path = f"{OUTPUT_DIR}/seasonal_index.csv"
seasonal_index.to_csv(seasonal_path, index=False)

print("\nSeasonal index (1.0 = average month):")
print(seasonal_index.to_string(index=False))

Download OK (437,366 bytes)
Sheets found: ['2026', '2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003']
Saved 12777 records | Year range: 2003 - 2026

Seasonal index (1.0 = average month):
 month  seasonal_index
     1        0.929165
     2        0.887446
     3        0.966480
     4        1.051458
     5        0.990557
     6        0.999165
     7        1.082145
     8        0.993799
     9        0.926826
    10        1.082289
    11        1.030723
    12        1.059947


---
## Module 4: Unified Feature Table

Merges all sources into a single flat table ready for LightGBM and DBSCAN.

**Output:** `features_unified.csv`
```
place_id | name | lat | lng | category | wikidata_id | name_en |
osm_tourism | osm_historic | popularity | wiki_pageviews_12m |
opening_hours | weather_code | pop | temp_min | temp_max |
seasonal_index | current_month | day_of_week | is_weekend
```

> **Note:** `popularity` (Wikipedia pageviews, 0-1) is your crowd proxy and LightGBM target
> until richer temporal sensor data arrives from JNTO.


In [11]:
df_master   = pd.read_csv(f"{OUTPUT_DIR}/locations_master.csv")
df_weather  = pd.read_csv(f"{OUTPUT_DIR}/weather_forecast.csv")
df_seasonal = pd.read_csv(f"{OUTPUT_DIR}/seasonal_index.csv")

current_month = datetime.now().month

# Latest weather snapshot for Kyoto
weather_row = df_weather.sort_values("date", ascending=False).iloc[0]

# Seasonal index for current month
s_val = df_seasonal[df_seasonal["month"] == current_month]["seasonal_index"].values
seasonal_val = float(s_val[0]) if len(s_val) else 1.0

df_features = df_master.copy()
df_features["weather_code"]   = weather_row.get("weather_code")
df_features["pop"]            = pd.to_numeric(weather_row.get("pop"), errors="coerce")
df_features["temp_min"]       = pd.to_numeric(weather_row.get("temp_min"), errors="coerce")
df_features["temp_max"]       = pd.to_numeric(weather_row.get("temp_max"), errors="coerce")
df_features["seasonal_index"] = seasonal_val
df_features["current_month"]  = current_month
df_features["day_of_week"]    = datetime.now().weekday()
df_features["is_weekend"]     = int(datetime.now().weekday() >= 5)
df_features["fetch_date"]     = datetime.now().isoformat()[:10]

keep_cols = [
    "place_id", "name", "lat", "lng", "category",
    "wikidata_id", "name_en", "osm_tourism", "osm_historic", "heritage",
    "address", "opening_hours",
    "popularity", "wiki_pageviews_12m",
    "has_wiki_data",
    "weather_code", "pop", "temp_min", "temp_max",
    "seasonal_index", "current_month", "day_of_week", "is_weekend",
    "fetch_date"
]
df_features = df_features[[c for c in keep_cols if c in df_features.columns]]

feature_path = f"{OUTPUT_DIR}/features_unified.csv"
df_features.to_csv(feature_path, index=False)

print(f"Unified feature table saved: {feature_path}")
print(f"Shape: {df_features.shape}")
print(f"Popularity coverage: {df_features['popularity'].gt(0).sum()} / {len(df_features)} locations")
df_features.head(10)


Unified feature table saved: ./kyoto_data/features_unified.csv
Shape: (1720, 24)
Popularity coverage: 618 / 1720 locations


,place_id,name,lat,lng,category,wikidata_id,name_en,osm_tourism,osm_historic,heritage,...,has_wiki_data,weather_code,pop,temp_min,temp_max,seasonal_index,current_month,day_of_week,is_weekend,fetch_date
0,511247ac03f4f7604059a3334300cb814140f00102f901...,二条城,35.014008,135.748537,tourism.sights,Q1013399,Nijō Castle,attraction,castle,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
1,51b8c7ce50daf66040598086a9ded0834140f00102f901...,仁和寺,35.029812,135.714150,tourism.sights,Q1202871,Ninna temple,attraction,heritage,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
2,5168f7a094faf6604059adf1678448844140f00102f901...,龍安寺,35.033463,135.718088,tourism.sights,Q587371,Ryōan-ji,attraction,heritage,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
3,51d328498112f8604059e95b716cbd874140f00102f901...,上賀茂神社,35.060468,135.752259,tourism.sights,Q700448,Kamigamo Shrine,NaN,heritage,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
4,51f2bda470b8f86040596ab050c1bc844140f00102f901...,下鴨神社,35.037010,135.772515,tourism.sights,Q701620,Shimogamo Shrine,NaN,monastery,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
5,51ea67825e58f76040591d40614b0f854140f00102f901...,金閣寺,35.039529,135.729537,tourism.sights,Q270983,Kinkaku-ji,NaN,heritage,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
6,51299fbd2dedf7604059aa3560527c7d4140f00102f901...,金堂,34.980356,135.747702,tourism.sights,Q107020573,Golden Hall,NaN,yes,2.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
7,51727d961f1af960405992e81852457f4140f00102f901...,清水寺,34.994303,135.784439,tourism.sights,Q221716,Kiyomizu-dera,attraction,yes,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
8,51ee3ddd50edf76040592c38e351857d4140f00102f901...,東寺,34.980631,135.747719,tourism.sights,Q107020573,Golden Hall,NaN,yes,2.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08
9,516928f69108f86040597a2e9f5df47e4140f00102f901...,西本願寺,34.991832,135.751046,tourism.sights,Q1146038,Nishi-Hongan-ji,NaN,heritage,1.0,...,1,101,20.0,NaN,NaN,0.96648,3,6,1,2026-03-08


---
## Summary

| File | Contents |
|------|----------|
| `geoapify_locations.csv` | Named Kyoto locations with coordinates and opening hours |
| `overpass_enrichment.csv` | Geoapify locations enriched with Wikidata IDs and OSM tags |
| `wikipedia_popularity.csv` | 12-month pageview counts and normalized popularity scores |
| `locations_master.csv` | Merged Geoapify + Overpass + Wikipedia table |
| `weather_forecast.csv` | JMA daily forecast for Kyoto |
| `jnto_visitor_stats.csv` | Monthly visitor counts by nationality (2003-present) |
| `seasonal_index.csv` | Monthly seasonal multipliers |
| `features_unified.csv` | **Final merged feature table — LightGBM and DBSCAN input** |

**Next step:** DBSCAN clustering on `features_unified.csv` to discover anchor vs orbit taxonomy from data.
